In [ ]:
%pip install fpdf2 Pillow -q

In [ ]:
import re
import os
from pathlib import Path
import s3fs
import glob
import pandas as pd
import numpy as np
import xarray as xr
import rasterio
import geopandas as gpd
import cartopy
import matplotlib.colors as colors
import matplotlib.cm as cmx
import matplotlib.pyplot as plt

import argparse
from shapely.geometry import Point
from IPython.display import clear_output
from collections import defaultdict
from datacube.utils.aws import configure_s3_access

from fpdf import FPDF
from PIL import Image

## Set up extended aus albers projection for cartopy figures

In [ ]:
"""
We rely on a custom Australian Albers projection here, as the PROJ/cartopy implementation of ESPG:9473/3577 has an upper latitude limit that is too low for the northern extent of the domain.
See https://epsg.io/3577 and https://epsg.io/9473 for information about each of these projections  
For our purposes, a projection based upon the GRS80 ellipsoid is sufficient to mimic either of these projection systems.
"""

aus_albers = cartopy.crs.AlbersEqualArea(
    central_longitude=132,
    standard_parallels=[-18, -36],
    globe=cartopy.crs.Globe(ellipse='GRS80')
)


### import tile files from gdata1

In [ ]:
# add link to tile geojson in gdata1
tiles_explorer = gpd.read_file(
    'gdata1/projects/fc-sub-annual/tc_check_temp/ga_ls_tc_pc_cyear_3-regions-aws-explorer.geojson'
)

#  filepath to the txt file from Brad:
tc_tiles = 'gdata1/projects/fc-sub-annual/tc_check_temp/comparison_summary_2024_all.txt'

In [ ]:
# Create a directory to save the maps
output_map_dir = 'gdata1/projects/fc-sub-annual/tc_check_temp/diff_maps/tile_diff_maps/'
os.makedirs(output_map_dir, exist_ok=True)

In [ ]:
bounds = tiles_explorer.total_bounds
print(bounds)
print(tiles_explorer.crs)

In [ ]:
f, ax = plt.subplots(figsize=(7, 7), subplot_kw={'projection': aus_albers}, constrained_layout=True)
ax.set_title('Tiles from GA Summary Grid (DEA Explorer)', fontsize=14)
ax.set_facecolor('#e6f2ff')
ax.add_feature(cartopy.feature.NaturalEarthFeature(category='physical', name='land', scale='10m',
                                                   facecolor='#ffffcc', edgecolor='#808080'))

ax.set_extent([111, 155, -45, -8], crs=cartopy.crs.PlateCarree())
ax.gridlines(lw=0.5, ls='--', color='#a0a0a0', draw_labels=['left', 'bottom'])

tiles_explorer.plot(ax=ax, facecolor='none', edgecolor='black', lw=0.5, transform=cartopy.crs.CRS(f'{tiles_explorer.crs}'), zorder=2)

In [ ]:
with Path(tc_tiles).open('r', encoding='utf-8') as f:
    print(f.read())

In [ ]:
#parse tile ids from txt file and sace as txt file
def extract_tile_codes(textfile_path):
    pattern = re.compile(r'^File:\s+.*?(x\d+y\d+)', re.IGNORECASE)
    tile_ids = []
    file_paths = []
    seen = set()

    with Path(textfile_path).open('r', encoding='utf-8') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                tile_id = match.group(1)
                file_path = str(line).split('File:')[1].strip()
                if tile_id not in seen:
                    seen.add(tile_id)
                    tile_ids.append(tile_id)
                    file_paths.append(file_path)
    return tile_ids, file_paths

In [ ]:
tile_id_list, tile_id_paths = extract_tile_codes(tc_tiles)
print(tile_id_list)
print(len(tile_id_list))

In [ ]:
#use tile ids to  create geojson with footprints of tiles in the id list
selected_tiles = tiles_explorer[tiles_explorer['region_code'].isin(tile_id_list)]


In [ ]:
f, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': aus_albers}, constrained_layout=True)
ax.set_title('Tiles flagged as different for TCP 2024 runs', fontsize=14)
ax.set_facecolor('#e6f2ff')
ax.add_feature(cartopy.feature.NaturalEarthFeature(category='physical', name='land', scale='10m',
                                                   facecolor='#ffffcc', edgecolor='#808080'))

ax.set_extent([111, 155, -45, -8], crs=cartopy.crs.PlateCarree())
ax.gridlines(lw=0.5, ls='--', color='#a0a0a0', draw_labels=['left', 'bottom'])

selected_tiles.plot(ax=ax, facecolor='none', edgecolor='red', lw=1.5, transform=cartopy.crs.CRS(f'{selected_tiles.crs}'), zorder=2)

plt.savefig('gdata1/projects/fc-sub-annual/tc_check_temp/diff_maps/tiles_flagged_as_different_tcp2024.png', dpi=150)
plt.show()

In [ ]:
len(selected_tiles)

## investigate some individual tiles to see pixel-absed differences

Note that most of the functions in this section are taken from Brad Greer's fantastic `comparison.py` script and modified to work in this notebook. If they are broken, it's my fault.

In [ ]:
def get_tiles(root_dir: str) -> set:
    clear_output(wait=True)
    unique_pairs = set()
    orig_paths = S3FS.glob(f"{root_dir}/*/*")

    for p in orig_paths:
        # Extract only the last two components, e.g. x34/y48
        x, y = p.rsplit("/", 2)[-2:]
        unique_pairs.add(f"{x}/{y}")
    
    print(f"Retrieved tiles from: {root_dir}\n")
    return unique_pairs

In [ ]:
def get_diff_tiles(dir_1:str, dir_2:str, target_tiles_fname: str, output_dir:str):
    orig_paths = get_tiles(dir_1)

    for i in target_tiles_fname:
        clear_output(wait=True)
        x, y = i[20:23], i[23:26]
        print(f"Processing tile: {x}/{y}")
        if f"{x}/{y}" in orig_paths:
            cogs_1 = f"{dir_1}/{x}/{y}/2024--P1Y/{i}"
            cogs_2 = f"{dir_2}/{x}/{y}/2024--P1Y/{i}"

            print(cogs_2)
            try:
                with xr.open_dataset(cogs_1, engine='rasterio') as ds1, \
                xr.open_dataset(cogs_2, engine='rasterio') as ds2:
                    diff = (ds1.to_array() - ds2.to_array()).squeeze()

                    output_fname = os.path.basename(cogs_1).replace('.tif', '_diff.tif')
                    output_path = os.path.join(output_dir, output_fname)

                    diff.rio.to_raster(
                        output_path,
                        driver='COG',
                        compress='deflate'
                    )
            except Exception as e:
                print(f"Error processing tile {i}: {e}")



In [ ]:
S3FS = s3fs.S3FileSystem(anon=True)
configure_s3_access(cloud_defaults=True, aws_unsigned=False)

In [ ]:
orig_run = 's3://dea-public-data/derivative/ga_ls_tc_pc_cyear_3/2-0-0'
test_run = 's3://dea-public-data-dev/derivative/ga_ls_tc_pc_cyear_3_test/2-0-0'

diff_tif_dir = 'gdata1/projects/fc-sub-annual/tc_check_temp/diff_tifs/'

In [ ]:
# Check if all diff tifs already exist before generating them
expected_diff_files = [
    os.path.join(diff_tif_dir, os.path.basename(p).replace('.tif', '_diff.tif'))
    for p in tile_id_paths
]

# Check if all files in the list exist
all_files_exist = all(os.path.exists(f) for f in expected_diff_files)

if all_files_exist:
    print("All difference TIFF files already exist. Skipping generation.")
else:
    print("Some difference TIFF files are missing. Generating files...")
    get_diff_tiles(orig_run, test_run, tile_id_paths, diff_tif_dir)


### load the diff tifs

In [ ]:
diff_dir = 'gdata1/projects/fc-sub-annual/tc_check_temp/diff_tifs/'
diff_files = glob.glob(f"{diff_dir}/*_diff.tif")

records = []
for f in diff_files:
    ds = xr.open_dataset(f, engine='rasterio')
    arr = ds.band_data.values
    nonzero_mask = arr != 0
    nonzero_count = np.count_nonzero(nonzero_mask)
    unique_nonzero = set(np.unique(arr[nonzero_mask]))
    # Extract tile id (e.g. x23y43) from filename
    tile_match = re.search(r'x\d+y\d+', f)
    tile_id = tile_match.group(0) if tile_match else None
    # Extract product (e.g. bright_pc_10, green_pc_50, etc.)
    prod_match = re.search(r'final_(.*?)_diff\.tif', f)
    product = prod_match.group(1) if prod_match else None
    records.append({
        'tile_id': tile_id,
        'product': product,
        'nonzero_pixel_count': nonzero_count,
        'unique_nonzero_values': unique_nonzero
    })
    ds.close()

df = pd.DataFrame(records)
df

In [ ]:
# Create a new dataframe that only includes tiles with more than one non-zero pixel
df_multiple_diffs = df[df['nonzero_pixel_count'] > 1].copy()

df_multiple_diffs

In [ ]:
gdf_records = []

for f in diff_files:
    ds = xr.open_dataset(f, engine='rasterio')
    arr = ds.band_data.squeeze().values
    nonzero_mask = arr != 0
    # Get indices of non-zero pixels
    rows, cols = np.where(nonzero_mask)
    # Get pixel values
    pixel_values = arr[rows, cols]
    # Get transform and CRS
    transform = ds.rio.transform()
    crs = ds.rio.crs
    # Extract tile id and product
    tile_match = re.search(r'x\d+y\d+', f)
    tile_id = tile_match.group(0) if tile_match else None
    prod_match = re.search(r'final_(.*?)_diff\.tif', f)
    product = prod_match.group(1) if prod_match else None
    # Convert pixel indices to coordinates
    for r, c, val in zip(rows, cols, pixel_values):
        x, y = transform * (c + 0.5, r + 0.5) # center of pixel
        gdf_records.append({
            'geometry': Point(x, y),
            'tile_id': tile_id,
            'product': product,
            'pixel_value': val
        })
    ds.close()

gdf = gpd.GeoDataFrame(gdf_records, crs=crs)
gdf['pixel_value'] = gdf['pixel_value'].fillna('nan')
gdf.to_file('gdata1/projects/fc-sub-annual/tc_check_temp/diff_pixels.geojson', driver='GeoJSON')

In [ ]:
gdf.head()

In [ ]:
f, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': aus_albers}, constrained_layout=True)
ax.set_title('Location of non-zero difference pixels, with pixel values shown', fontsize=14)
ax.set_facecolor('#e6f2ff')
ax.add_feature(cartopy.feature.NaturalEarthFeature(category='physical', name='land', scale='10m',
                                                   facecolor='#ffffcc', edgecolor='#808080'))

ax.set_extent([111, 155, -45, -8], crs=cartopy.crs.PlateCarree())
ax.gridlines(lw=0.5, ls='--', color='#a0a0a0', draw_labels=['left', 'bottom'])

# Plot the non-zero pixel locations
cmap = colors.ListedColormap(cmx.get_cmap("Dark2").colors[:3])

gdf.plot(ax=ax,
         column='pixel_value',
         categorical=True,
         markersize=10,
         cmap=cmap,
         transform=cartopy.crs.CRS(f'{gdf.crs}'),
         legend=True,
         legend_kwds={'title': 'Pixel Value',
                      'loc': 'upper right',
                      'bbox_to_anchor': (1, 1)},
         zorder=3)

plt.savefig('gdata1/projects/fc-sub-annual/tc_check_temp/diff_maps/pixels_flagged_as_different_tcp2024.png', dpi=150)


In [ ]:
# Get unique tile IDs from the dataframe with multiple differences
tiles_to_plot = df_multiple_diffs['tile_id'].unique()

# Convert NaN values in 'pixel_value' to a string to ensure they are plotted
gdf['pixel_value'] = gdf['pixel_value'].fillna('NaN')

# Loop through the tiles and create a map for each
for target_tile_id in tiles_to_plot:
    
    # Get the footprint for the target tile
    tile_footprint = tiles_explorer[tiles_explorer['region_code'] == target_tile_id]
    
    # Check if the footprint exists before plotting
    if not tile_footprint.empty:
        # Get the non-zero pixels within that tile
        points_in_tile = gdf[gdf['tile_id'] == target_tile_id]

        # Skip if there are no points for this tile in the gdf
        if points_in_tile.empty:
            print(f"No difference points found for tile_id: {target_tile_id}")
            continue

        # Get the bounds of the tile's footprint to set the map extent
        bounds = tile_footprint.total_bounds
        buffer = 0.05 # degrees
        # Correct order for cartopy extent: [minx, maxx, miny, maxy]
        map_extent = [bounds[0] - buffer, bounds[2] + buffer, bounds[1] - buffer, bounds[3] + buffer]

        # Create the plot
        f, ax = plt.subplots(figsize=(8, 6), subplot_kw={'projection': aus_albers}, constrained_layout=True)
        f.suptitle(f'Non-zero difference pixels for tile {target_tile_id}', fontsize=14)
        ax.set_title(f'Number of non-zero pixels in {target_tile_id}: {len(points_in_tile)}', fontsize=10)
        ax.set_facecolor('#e6f2ff')
        ax.add_feature(cartopy.feature.NaturalEarthFeature(category='physical', name='land', scale='10m',
                                                           facecolor='#ffffcc', edgecolor='#808080'))

        # Set the extent to the tile's bounds plus a buffer
        ax.set_extent(map_extent, crs=cartopy.crs.CRS(f'{tile_footprint.crs}'))
        ax.gridlines(lw=0.5, ls='--', color='#a0a0a0', draw_labels=['left', 'bottom'])

        # Plot the tile footprint by reprojecting it first
        tile_footprint.to_crs(aus_albers).plot(ax=ax,
                                              facecolor='none',
                                              edgecolor='black',
                                              lw=1.0,
                                              zorder=2)

        # Plot the non-zero pixel locations, colored by value
        points_in_tile.to_crs(aus_albers).plot(ax=ax,
                            column='pixel_value',
                            categorical=True, 
                            cmap=cmap, 
                            markersize=15,
                            zorder=3,
                            legend=True,     
                            legend_kwds={
                                'title': "Pixel Value",
                                'loc': 'upper left',
                                'bbox_to_anchor': (1.15, 1) # Place legend outside plot
                            })

        # ----- create and add inset map-----#
        inset_ax = f.add_axes([0.85, 0.1, 0.25, 0.25], projection=aus_albers)
        inset_ax.set_extent([111, 155, -45, -8], crs=cartopy.crs.PlateCarree())
        inset_ax.add_feature(cartopy.feature.NaturalEarthFeature(category='physical', name='land', scale='50m',
                                                                 facecolor='#ffffcc', edgecolor='#808080', lw=0.5))
        tile_footprint.to_crs(aus_albers).plot(ax=inset_ax,
                                              facecolor='red',
                                              edgecolor='red',
                                              lw=3,
                                              zorder=4)
        
        # Save the figure
        output_path = os.path.join(output_map_dir, f'{target_tile_id}_diff_map.png')
        plt.savefig(output_path, dpi=150, bbox_inches='tight')

    else:
        print(f"No footprint found for tile_id: {target_tile_id}")

### Generate summary report


In [ ]:
def dataframe_to_pdf(pdf, df):
    df_str = df.astype(str)
    
    pdf.set_font("Helvetica", "B", 6)
    col_width = pdf.epw / len(df_str.columns)
    for header in df_str.columns:
        pdf.cell(col_width, 6, header, border=1, align='C')
    pdf.ln()


    pdf.set_font("Helvetica", "", 5)
    for i in range(len(df_str)):
        for col in df_str.columns:
            pdf.cell(col_width, 5, df_str[col].iloc[i], border=1, align='C')
        pdf.ln()


In [ ]:
# Setup PDF document
pdf = FPDF(orientation="P", unit="mm", format="A4")
pdf.set_auto_page_break(auto=True, margin=15)
pdf_output_path = "gdata1/projects/fc-sub-annual/tc_check_temp/TC_2024_Difference_Report.pdf"

# Add Title Page
pdf.add_page()
pdf.set_font("Helvetica", "B", 24)
pdf.cell(0, 20, "Tasselled Cap Percentile 2024 Difference Report", ln=True, align="C")
pdf.set_font("Helvetica", "", 12)
pdf.cell(0, 10, f"Generated on: {pd.Timestamp.now().strftime('%Y-%m-%d')}", ln=True, align="C")

# Add the full DataFrame (df)
pdf.add_page()
pdf.set_font("Helvetica", "B", 14)
pdf.cell(0, 10, "All Tiles with Non-Zero Pixel Differences", ln=True)
dataframe_to_pdf(pdf, df)

# Add the filtered DataFrame (df_multiple_diffs)
pdf.add_page()
pdf.set_font("Helvetica", "B", 14)
pdf.cell(0, 10, "Tiles with More Than One Non-Zero Pixel", ln=True)
dataframe_to_pdf(pdf, df_multiple_diffs)

# Add map images
map_dir = 'gdata1/projects/fc-sub-annual/tc_check_temp/diff_maps/'
image_files = sorted(glob.glob(f"{map_dir}/*.png"))

if image_files:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "Tiles and Pixels flagged as different between runs", ln=True)

    # Define image placement properties
    img_margin = 5
    max_img_height = (pdf.eph - img_margin) / 2
    
    for i, img_path in enumerate(image_files):
        # Add a new page for every two images (at the start of a pair)
        if i % 2 == 0:
            pdf.add_page()
            y_pos = pdf.get_y()

        # Get image dimensions to maintain aspect ratio
        with Image.open(img_path) as img:
            img_width, img_height = img.size
            aspect_ratio = img_height / img_width
        
        new_h = max_img_height
        new_w = new_h / aspect_ratio
        
        x_pos = (pdf.w - new_w) / 2
        pdf.image(img_path, x=x_pos, y=y_pos, w=new_w, h=new_h)

        if i % 2 == 0:
            y_pos += new_h + img_margin # Move down for next image
        
    

# Add tile map images
map_dir = 'gdata1/projects/fc-sub-annual/tc_check_temp/diff_maps/tile_diff_maps/'
image_files = sorted(glob.glob(f"{map_dir}/*_diff_map.png"))

if image_files:
    pdf.add_page()
    pdf.set_font("Helvetica", "B", 14)
    pdf.cell(0, 10, "Tiles and Pixels flagged as different between runs", ln=True)

    # Define image placement properties
    img_margin = 5
    max_img_height = (pdf.eph - img_margin) / 2
    
    for i, img_path in enumerate(image_files):
        # Add a new page for every two images (at the start of a pair)
        if i % 2 == 0:
            pdf.add_page()
            y_pos = pdf.get_y()

        # Get image dimensions to maintain aspect ratio
        with Image.open(img_path) as img:
            img_width, img_height = img.size
            aspect_ratio = img_height / img_width
        
        new_h = max_img_height
        new_w = new_h / aspect_ratio
        
        x_pos = (pdf.w - new_w) / 2
        pdf.image(img_path, x=x_pos, y=y_pos, w=new_w, h=new_h)

        if i % 2 == 0:
            y_pos += new_h + img_margin # Move down for next image

# Save the PDF
try:
    pdf.output(pdf_output_path)
    print(f"Successfully generated PDF report: {pdf_output_path}")
except Exception as e:
    print(f"Failed to generate PDF: {e}")